In [1]:
import numpy as np
def funkcja(punkty, prog = 0.1, iteracje = 1000):
    lista = []
    for i in range(iteracje):
        idx = np.random.choice(len(punkty), 3, replace = False)
        p1, p2, p3 = punkty[idx]
        n = np.cross(p2 - p1, p3-p1)
        n = n / np.linalg.norm(n)
        d = -np.dot(n, p1)
        odl = np.abs(punkty @ n + d)
        inl = np.where(odl < prog) [0]
        if len(inl) > len(lista):
            lista = inl
            lista_n, lista_d = n, d 

    return lista_n, lista_d, lista



In [10]:
import csv 
import numpy as np 
def plik(sciezka):
    punkty = []
    with open (sciezka) as f:
        reader = csv.reader(f, delimiter=' ')
        for wiersz in reader:
            x, y, z = map(float, wiersz)
            punkty.append([x, y, z])
    return np.array(punkty)

folder = r'd:\Ksentun Data\elektronika i telekomunikacja\Laboratoria IO\Laboratoria-1'

pozioma = plik(folder + r'\plaska_pozioma.xyz')
pionowa = plik(folder + r'\plaska_pionowa.xyz')
walec   = plik(folder + r'\cylindr.xyz')

walec   = walec   + [-20, -20, 0]   
pozioma = pozioma + [0, 0, 10]       

punkty = np.vstack((pozioma, pionowa, walec))

punkty = np.vstack((pozioma, pionowa, walec))   

print(pozioma.shape, pionowa.shape, walec.shape)
print(punkty.shape)
print(punkty[:3])

for nazwa, chmura in [('pozioma', pozioma), ('pionowa', pionowa), ('walec', walec)]:
    print(nazwa, 'min:', chmura.min(axis=0).round(1), 'max:', chmura.max(axis=0).round(1))


(100, 3) (100, 3) (100, 3)
(300, 3)
[[ 3.130679  3.628019 21.      ]
 [ 6.133326 12.829405 21.      ]
 [ 7.512544  3.708604 21.      ]]
pozioma min: [ 3.1  3.1 21. ] max: [15.9 15.9 21. ]
pionowa min: [0.  4.  0.1] max: [13.   4.   4.9]
walec min: [-25.  -25.    0.1] max: [-15. -15.  10.]


In [11]:
from sklearn.cluster import KMeans 
kmeans = KMeans(n_clusters=3, n_init =10, random_state=0)
etykiety = kmeans.fit_predict(punkty)
print(etykiety)
print(np.bincount(etykiety))


[2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1]
[100 100 100]


In [12]:
for k in range(3):
    chmura = punkty[etykiety == k]
    n, d, inli = funkcja(chmura)
    srednia = np.mean(np.abs(chmura @ n + d))
    print(f'Chura {k}:')
    print(' wektor normalny:', n.round(3))
    print(' średnia odległość:', round(srednia, 3))

    if srednia > 0.5:
        print('-> to nie jest płaszczyzna')
    elif abs(n[2]) > 0.9:
        print(' -> płaszczyzna Pozioma')
    elif abs(n[2]) < 0.1:
        print(' -> płaszczyzna Pionowa')
    

Chura 0:
 wektor normalny: [0. 1. 0.]
 średnia odległość: 0.0
 -> płaszczyzna Pionowa
Chura 1:
 wektor normalny: [ 0.12   0.993 -0.008]
 średnia odległość: 4.986
-> to nie jest płaszczyzna
Chura 2:
 wektor normalny: [0. 0. 1.]
 średnia odległość: 0.0
 -> płaszczyzna Pozioma


In [13]:
from sklearn.cluster import DBSCAN
import pyransac3d as pyrsc

dbscan = DBSCAN(eps = 3, min_samples= 5)
etykiety_db = dbscan.fit_predict(punkty)
print(np.unique(etykiety_db, return_counts=True))

for k in np.unique(etykiety_db):
    if k == -1:
        continue
    chmura = punkty[etykiety_db == k]
    plaszczyzna = pyrsc.Plane()
    rownanie, inliers = plaszczyzna.fit(chmura, thresh=0.1, maxIteration = 1000)
    n = np.array(rownanie[:3])
    d = rownanie[3]
    dlugosc = np.linalg.norm(n)
    n,d = n/dlugosc, d/dlugosc
    srednia = np.mean(np.abs(chmura @ n +d))
    print(f'Chmura {k}:')
    print('  wektor normalny:', n.round(3))
    print('  średnia odległość:', round(srednia, 3))
    if srednia > 0.5:
        print('  -> to nie jest płaszczyzna')
    elif abs(n[2]) > 0.9:
        print('  -> płaszczyzna pozioma')
    elif abs(n[2]) < 0.1:
        print('  -> płaszczyzna pionowa')




(array([0, 1, 2]), array([100, 100, 100]))
Chmura 0:
  wektor normalny: [0. 0. 1.]
  średnia odległość: 0.0
  -> płaszczyzna pozioma
Chmura 1:
  wektor normalny: [-0.  1.  0.]
  średnia odległość: 0.0
  -> płaszczyzna pionowa
Chmura 2:
  wektor normalny: [0.849 0.528 0.009]
  średnia odległość: 4.995
  -> to nie jest płaszczyzna
